# Chapter 12 — Optimize the Instructions

**Book alignment:** DSPy From First Principles, Chapter 12

**Question this notebook isolates:** Does an instruction-search headline gain decompose into one fixed case while breaking the same sentence the same way?

In [ ]:
import inspect
from pathlib import Path
import random
import sys

random.seed(13)


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy  # imported and constructed only; compile is never executed against a model

from common.data import canonical_split
from common.dspy_api import no_call_lm
from common.dspy_program import dspy_editorial_metric_v1
from common.metrics import editorial_metric

## Decompose the recorded gain case by case

The chapter's single-run +0.0298 clears the ±0.015 noise floor — but an aggregate is a summary. Replay the per-case arithmetic over the 11 development cases and check that it closes exactly.

In [ ]:
N_DEV = 11
# Recorded (baseline, candidate) v1 pairs for the five cases that moved; rest unchanged.
moves = {
    "ed-029": (0.650, 1.000),
    "ed-033": (0.818, 0.927),
    "ed-035": (1.000, 0.967),
    "ed-003": (0.877, 0.846),
    "ed-038": (1.000, 0.933),
}
contribs = {case: (cand - base) / N_DEV for case, (base, cand) in moves.items()}
total = sum(contribs.values())

[(case, round(delta, 4), round(c, 4)) for case, delta, c in
 ((case, moves[case][1] - moves[case][0], contribs[case]) for case in moves)]
print("total:", round(total, 4))

In [ ]:
assert abs(total - 0.0298) < 1e-4
assert contribs["ed-029"] > total  # one fixed case contributes more than the headline
assert abs(sum(c for case, c in contribs.items() if case != "ed-029") - (-0.0020)) < 1e-4

print(f"ed-029 contributes {contribs['ed-029']:.4f} alone; everything else nets {total - contribs['ed-029']:.4f}")
print("MIPROv2 fixed one case, made three slightly worse, and got lucky on a fourth")

## The same exploit, and why the dev score is selection evidence

Unlike BootstrapFewShot, MIPROv2 selects against the development set — verify that `valset` is part of its compile contract. Then confirm the entire v1/v2 divergence is the ed-035 sentence, broken the same way as in Chapter 11.

In [ ]:
split = canonical_split()
optimizer = dspy.MIPROv2(
    metric=dspy_editorial_metric_v1,
    prompt_model=no_call_lm(),  # constructed only; never called, so no network
    task_model=no_call_lm(),  # constructed only; never called, so no network
    auto=None,
    num_candidates=2,
    seed=13,
)
compile_params = inspect.signature(optimizer.compile).parameters

dev_by_id = {c.case_id: c for c in split.dev}
ed035 = dev_by_id["ed-035"]
bad = ed035.reference_rewrite.replace("regularly ", "")
v1_bad = editorial_metric(ed035, bad, version="v1").score
v2_bad = min(v1_bad * 0.30, 0.30)

gap_per_case = (1.0 - v2_bad)  # 1.000 -> v2 on ed-035
metric_gap = (0.0298 - (-0.0308))  # recorded single-run v1 delta minus v2 delta

({
    "compile_takes_valset": "valset" in compile_params,
    "v1_bad": round(v1_bad, 4),
    "v2_bad": round(v2_bad, 4),
    "gap_per_case_over_11": round(gap_per_case / 11, 4),
    "recorded_metric_gap": round(metric_gap, 4),
})

In [ ]:
assert "valset" in compile_params  # dev evidence enters search: score is selection evidence
assert abs(v1_bad - 0.9667) < 1e-4 and abs(v2_bad - 0.29) < 1e-9
rounded_gap = (0.967 - 0.300) / 11  # book's reported per-case readings: 0.667 / 11
assert abs(round(v1_bad, 3) - 0.967) < 1e-9  # independently recomputed v1 matches the book
assert v2_bad <= 0.30  # replay sits at the violation cap the book reports as 0.300
assert abs(rounded_gap - metric_gap) < 1e-4  # one sentence explains the whole divergence
assert "ed-035" in split.dev_ids and "ed-035" not in split.holdout_ids

print("the entire v1/v2 divergence is ed-035 dropping 'regularly' — again")
print("two mechanisms, one shared objective, one shared exploit")

## What we earned

Instruction search genuinely fixed ed-029 — a real, reproducible, mechanistic improvement — while the headline +0.03 decomposed to one case plus noise. And a second optimizer with different evidence and far more compute deleted `regularly` exactly like the first: the exploit lives in the objective, and a larger search only expresses it more thoroughly.

Notebook 13 / Chapter 13 replaces the scalar with language: what changes when the optimizer receives feedback in words rather than a score?